# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamyaMuthumariappan/FlyRankAI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RamyaMuthumariappan/FlyRankAI.git"
REPO_DIR = "FlyRankAI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)

Working dir: /content/FlyRankAI/FlyRankAI
Starter data found. You're ready.
Rows, columns: (30000, 44)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.**

Lane 2's real question is "which of ~30,000 pages does the editor open first, and what should
they do with it" — not "is this page declining, yes/no." A yes/no classifier alone doesn't answer
that: even a perfectly-labeled declining/not-declining split still leaves an editor with time for
~20-30 pages a week needing an *ordering*, not a bucket. So the task type is ranking/scoring:
produce a continuous priority score, sort all ~30,000 pages by it, and hand the top slice to the
editor as a worklist.

A classifier still shows up **inside** this — `is_declining_label` is one input signal feeding the
score — but it isn't the deliverable. The deliverable is the ordering that comes out once decline
signal is combined with severity and visibility context. (Matches the framing skill's own table:
"Which ones first?" → Ranking / scoring.)

In [15]:
task_type = "Ranking / scoring"
print("Lane: Lane 2 — Refresh / Content Opportunity Scoring")
print("Task type:", task_type)
print("Why: the deliverable is an ORDERED worklist of ~30-50 pages for a human editor,")
print("not a single yes/no label. is_declining_label can feed the score as one signal,")
print("but the score itself is what gets ranked and capped at editor capacity.")


Lane: Lane 2 — Refresh / Content Opportunity Scoring
Task type: Ranking / scoring
Why: the deliverable is an ORDERED worklist of ~30-50 pages for a human editor,
not a single yes/no label. is_declining_label can feed the score as one signal,
but the score itself is what gets ranked and capped at editor capacity.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no ground-truth "this page needed review, and the editor confirmed it helped" label
anywhere in this dataset — no editor has gone through and tagged real outcomes. So this is a
**proxy problem from the start**, and I want to be precise about which proxy I'm using and why
it isn't a clean observed outcome.

The closest thing to a measured signal is `is_declining_label` (`trend_direction == "down"`),
which is derived from `trend_pct` — the real percentage change in impressions between the last
30 days and the previous 30 days, thresholded at a -20% swing. That's a genuinely **observed**
change in real traffic — but a **defined threshold** on top of it, not an independently-measured
outcome like "an editor reviewed this page and it needed a refresh." I'm treating it as a real
but imperfect proxy, not ground truth, and I will never use `trend_direction` / `trend_pct` as a
model *feature* later — only as the thing the proxy label is built from (per the leakage rule
from week 1).

For this week, the target I'm sketching toward is a composite `priority_tier` (high / medium /
low) built from:
- the decline signal (`is_declining_label`), used only to construct the proxy, never downstream
  as a feature;
- severity context: how visible the page still is (`avg_position`, `impressions_last_30d`) and
  how stale it is (`days_since_last_update`).

This tier is a stand-in for the continuous priority *score* I'd build properly in later weeks.

In [16]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

page1_decay = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["is_declining_label"] == 1)
stale_visible = (df["days_since_last_update"] > 60) & (df["impressions_last_30d"] > 50)

df["priority_tier"] = "low"
df.loc[df["is_declining_label"] == 1, "priority_tier"] = "medium"
df.loc[page1_decay | stale_visible, "priority_tier"] = "high"

print(df["priority_tier"].value_counts())
print()
print("Reminder: trend_direction/trend_pct built this proxy label — they are NEVER model features.")


priority_tier
high      12694
low       10766
medium     6540
Name: count, dtype: int64

Reminder: trend_direction/trend_pct built this proxy label — they are NEVER model features.


## 3. Success metric

*One metric you can defend. What number means "good"?*

**precision@K**, with K matched to what an editor can actually work through — the card's own
"top 30-50 page worklist" framing. Concretely: of the top 30 pages by priority score, what
fraction are genuinely high-priority by the proxy definition?

I'm choosing precision@K over plain accuracy or ROC-AUC because nobody reviews all 30,000 pages
— the entire value of this system lives in the *top* of the queue. A model that's 95% "accurate"
overall but wrong on 15 of my top 30 is useless to the editor; a model that's mediocre on the
bottom 29,970 pages nobody will ever open is irrelevant to them. Recall matters less here than
precision, because a missed page just waits another week, while a wasted editor-hour on a false
alarm is a cost paid immediately (the asymmetric-cost argument from week 1).

I'll also do the lane guide's recommended by-hand review of the top 20 — a human sanity check
that precision@K alone can't catch (e.g. 30 technically-correct-but-editorially-useless pages).

In [17]:


K = 30
naive_rank = df[df["is_declining_label"] == 1].sort_values("avg_position").head(K)

# "hit" = the page also clears a second, independent severity check (stale-but-visible)
hits = (naive_rank["days_since_last_update"] > 30).sum()
print(f"precision@{K} (illustrative, naive rank vs a second severity check): {hits}/{K} = {hits/K:.0%}")
print("Placeholder computation to prove the metric mechanics work on this data —")
print("the real precision@K needs an actual scored/ranked model (later weeks).")


precision@30 (illustrative, naive rank vs a second severity check): 8/30 = 27%
Placeholder computation to prove the metric mechanics work on this data —
the real precision@K needs an actual scored/ranked model (later weeks).


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item** — a single published page, identified by `content_id`, nested
under one `client_id`. 30,000 rows across 32 clients. This is already the right grain for
Lane 2: the editor reviews *pages*, not clients or search queries, so no reshaping or
aggregation is needed before I can rank.

In [18]:
unit_cols = ["content_id", "client_id", "content_type", "avg_position",
             "impressions_last_30d", "days_since_last_update", "priority_tier"]
df[unit_cols].head(10)

,content_id,client_id,content_type,avg_position,impressions_last_30d,days_since_last_update,priority_tier
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,578,20,medium
1,content_a1fb4e703a9e,client_4e07408562,keyword article,20.3,2501,25,medium
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,36.5,2382,20,medium
3,content_331d6c4de07b,client_19581e27de,keyword article,6.2,3626,22,low
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,44.0,4211,14,medium
5,content_d4084a4bc775,client_f369cb89fc,keyword article,8.5,617,20,high
6,content_9a34b442b552,client_8722616204,keyword article,7.0,1,20,high
7,content_a63219c6e95a,client_19581e27de,keyword article,21.2,636,22,low
8,content_5e6c160719bc,client_6208ef0f77,keyword article,46.0,5696,20,medium
9,content_c27558df2b0c,client_19581e27de,keyword article,4.9,252,104,high


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement can catch ONE shape of "needs review" — e.g. "page-one AND declining"
covers 24.4% of the catalogue — but the lane guide names at least five different shapes:
stale-but-visible, declining-with-demand, thin-but-visible, page-one decay, CTR/engagement
outliers. These overlap only partially, and hand-picking weights across five-plus signals, per
content type, per client, so the top 30 by score are actually the 30 most worth an editor's time,
is exactly the "many signals, tangled, shifting over time" pattern the framing skill flags as
ML's home turf. A fixed rule needs someone to guess the right combination and threshold up front
and never revisit it; a scored/learned combination can be tuned against precision@K and
re-weighted as the catalogue's behavior shifts over time.

That said — the transparent rule-based baseline (Lane 2's own "building-baselines" step) is
still the right *first* thing to build, precisely so I have something honest to beat before
reaching for a model.

In [19]:
stale_visible = (df["days_since_last_update"] > 60) & (df["impressions_last_30d"] > 50)
page1_decay = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["is_declining_label"] == 1)

print("stale-but-visible pages:", stale_visible.sum())
print("page-one decay pages:", page1_decay.sum())
print("overlap (both rules agree):", (stale_visible & page1_decay).sum())
print("stale-but-visible ONLY (a page-one-decay rule alone would miss these):",
      (stale_visible & ~page1_decay).sum())
print()
print("These are mostly different pages — no single rule covers both shapes of risk.")


stale-but-visible pages: 7076
page-one decay pages: 7311
overlap (both rules agree): 1693
stale-but-visible ONLY (a page-one-decay rule alone would miss these): 5383

These are mostly different pages — no single rule covers both shapes of risk.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.